### Inspect scraped data_files

In [2]:
import pandas as pd
from itertools import combinations

In [3]:
logboek_df = pd.read_csv('logboek.csv', sep=';')
countries = logboek_df["land"].tolist()
leagues = logboek_df["competitienaam"].tolist()
seasons = logboek_df["jaar"].tolist()

In [4]:
print(countries)

['england', 'mexico', 'japan', 'iran', 'south-korea', 'australia', 'morocco', 'senegal', 'nigeria', 'algeria', 'egypt', 'ivory-coast', 'usa', 'canada', 'panama', 'argentina', 'brazil', 'colombia', 'uruguay', 'ecuador', 'paraguay', 'austria', 'belgium', 'cyprus', 'czech-republic', 'france', 'germany', 'hungary', 'ireland', 'italy', 'luxembourg', 'malta', 'netherlands', 'northern-ireland', 'norway', 'poland', 'portugal', 'romania', 'russia', 'scotland', 'slovakia', 'spain', 'switzerland', 'turkey', 'wales', 'croatia', 'denmark', 'ukraine', 'serbia', 'sweden']


### checked up untill index 0

In [5]:
def filter_rounds(df):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str.startswith("Round ")].reset_index(drop=True)
    return return_df

In [6]:
def filter_playoffs(df, start_round_playoffs):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str[6:].astype(int) < start_round_playoffs].reset_index(drop=True)
    return return_df

In [7]:
def filter_on_date(df, start_date, keep_before = True):
    """ 
    Format start_date: "YYYY-MM-DD"
    """
    return_df = df.copy()
    start_date = pd.to_datetime(start_date)
    
    return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')
    if keep_before:
        return_df = return_df[(return_df["date"] < start_date)].reset_index(drop=True)
    else:
        return_df = return_df[(return_df["date"] >= start_date)].reset_index(drop=True)
    return return_df

In [8]:
def remove_round_prefix(df):
    return_df = df.copy()
    return_df["round"] = return_df["round"].apply(lambda x: x[6:])
    return return_df

In [71]:
def check_specific_match(df, team1, team2):
    return df[((df["team_home"] == team1) & (df["team_away"] == team2)) | ((df["team_home"] == team2) & (df["team_away"] == team1))]

In [9]:
def check_rounds(df):
    rounds = df["round"].unique().tolist()
    counted_rounds = df["round"].value_counts().to_dict()

    # check if all rounds appear the same amount of times
    rounds_appear_same_amount = len(set(counted_rounds.values())) == 1

    # check if rounds start with "Round"
    rounds_start_with_round = all(r.startswith("Round") for r in rounds)

    print(f"Rounds: {rounds}")
    print(f"Only normal rounds: {rounds_start_with_round}")
    print(f"All rounds appear same amount of times: {rounds_appear_same_amount}")

In [10]:
def check_teams(df):
    home_teams = df["team_home"].unique().tolist()
    away_teams = df["team_away"].unique().tolist()

    all_teams = set(home_teams) | set(away_teams)
    print(f"Teams: {all_teams}")
    print(f"Number of teams: {len(all_teams)}")

    # check if all teams appear in both home and away the same amount of times
    home_team_counts = df["team_home"].value_counts().to_dict()
    away_team_counts = df["team_away"].value_counts().to_dict()
    teams_appear_same_amount = all(home_team_counts.get(team, 0) == away_team_counts.get(team, 0) for team in all_teams)
    print(f"All teams appear same amount of times in home and away: {teams_appear_same_amount}")
    if teams_appear_same_amount:
        print(f"Each team appears {home_team_counts[home_teams[0]]} times in home and away")


In [11]:
def check_games(df):
    teams = sorted(set(df['team_home'].unique()) | set(df['team_away'].unique()))
    
    # Build symmetric n x n matrix
    matrix = pd.DataFrame(0, index=teams, columns=teams)
    
    for _, row in df.iterrows():
        h, a = row['team_home'], row['team_away']
        matrix.loc[h, a] += 1
        matrix.loc[a, h] += 1  # symmetric: home/away doesn't matter
    
    # Extract counts for all unique pairs (upper triangle)
    pair_counts = {
        (t1, t2): matrix.loc[t1, t2]
        for t1, t2 in combinations(teams, 2)
    }
    
    unique_counts = set(pair_counts.values())
    is_balanced = len(unique_counts) == 1 and 0 not in unique_counts
    
    print("=== Game Balance Check ===\n")
    print("Head-to-head matrix:")
    print(matrix.to_string())
    print()
    
    count_freq = {}
    for v in pair_counts.values():
        count_freq[v] = count_freq.get(v, 0) + 1
    
    if is_balanced:
        times = unique_counts.pop()
        print(f"Balanced: every pair plays exactly {times} time(s).")
    else:
        print(f"Not balanced. Distribution of matchup counts: {count_freq}")
        if 0 in unique_counts:
            missing = [f"{t1} vs {t2}" for (t1, t2), v in pair_counts.items() if v == 0]
            print(f"   Never played: {missing[:10]}{'...' if len(missing) > 10 else ''}")
        modal = max(set(pair_counts.values()), key=list(pair_counts.values()).count)
        odd = {f"{k[0]} vs {k[1]}": v for k, v in pair_counts.items() if v != modal}
        if odd:
            print(f"   Odd counts: {odd}")
    

In [79]:
# Checked untill index 10, egypt
index = 24
country = countries[index]
league = leagues[index]
season = seasons[index]
print(country)
print(league)
print(season)
filename = f"{country}_{league}_{season}.csv"
df = pd.read_csv(f"./scraped_data/{filename}")
df

czech-republic
chance-liga
2024-2025


,round,date,team_home,team_away,full_text
0,Final,01.06.2025,Chrudim,FK Pardubice,01.06.2025 | Chrudim | FK Pardubice | Advancin...
1,Final,01.06.2025,Pen,Dukla Prague,01.06.2025 | Pen | Dukla Prague | Winner: Dukl...
2,Final,28.05.2025,FK Pardubice,Chrudim,28.05.2025 | FK Pardubice | Chrudim | 2 | 0
3,Final,28.05.2025,Vyskov,Dukla Prague,28.05.2025 | Vyskov | Dukla Prague | 0 | 0
4,Round 5,25.05.2025,Dukla Prague,Ceske Budejovice,25.05.2025 | Dukla Prague | Ceske Budejovice |...
...,...,...,...,...,...
275,Round 1,20.07.2024,Dukla Prague,Plzen,20.07.2024 | Dukla Prague | Plzen | 1 | 3
276,Round 1,20.07.2024,Bohemians,Ostrava,20.07.2024 | Bohemians | Ostrava | 2 | 1
277,Round 1,20.07.2024,Ceske Budejovice,Sigma Olomouc,20.07.2024 | Ceske Budejovice | Sigma Olomouc ...
278,Round 1,20.07.2024,Jablonec,Mlada Boleslav,20.07.2024 | Jablonec | Mlada Boleslav | 2 | 2...


In [80]:
# First do the checks, then do the filtering and removing of round prefix if necessary

check_rounds(df)
check_teams(df)
check_games(df)

Rounds: ['Final', 'Round 5', 'Round 4', 'Round 3', 'Semi-finals', 'Round 2', 'Round 1', 'Round 30', 'Round 29', 'Round 23', 'Round 28', 'Round 27', 'Round 26', 'Round 21', 'Round 25', 'Round 24', 'Round 22', 'Round 20', 'Round 19', 'Round 18', 'Round 8', 'Round 17', 'Round 15', 'Round 16', 'Round 14', 'Round 13', 'Round 12', 'Round 11', 'Round 10', 'Round 9', 'Round 6', 'Round 7']
Only normal rounds: False
All rounds appear same amount of times: False
Teams: {'Plzen', 'Karvina', 'Pen', 'Dukla Prague', 'Ceske Budejovice', 'Vyskov', 'Slovacko', 'Liberec', 'Sigma Olomouc', 'Hradec Kralove', 'Chrudim', 'Bohemians', 'Jablonec', 'Ostrava', 'Sparta Prague', 'FK Pardubice', 'Mlada Boleslav', 'Teplice', 'Winner: Hradec Kralove', 'Slavia Prague'}
Number of teams: 20
All teams appear same amount of times in home and away: False
=== Game Balance Check ===

Head-to-head matrix:
                        Bohemians  Ceske Budejovice  Chrudim  Dukla Prague  FK Pardubice  Hradec Kralove  Jablonec  Karvin

In [81]:
df_only_rounds = filter_rounds(df)

In [82]:
check_rounds(df_only_rounds)
check_teams(df_only_rounds)
check_games(df_only_rounds)

Rounds: ['Round 5', 'Round 4', 'Round 3', 'Round 2', 'Round 1', 'Round 30', 'Round 29', 'Round 23', 'Round 28', 'Round 27', 'Round 26', 'Round 21', 'Round 25', 'Round 24', 'Round 22', 'Round 20', 'Round 19', 'Round 18', 'Round 8', 'Round 17', 'Round 15', 'Round 16', 'Round 14', 'Round 13', 'Round 12', 'Round 11', 'Round 10', 'Round 9', 'Round 6', 'Round 7']
Only normal rounds: True
All rounds appear same amount of times: False
Teams: {'Plzen', 'Karvina', 'Dukla Prague', 'Ceske Budejovice', 'Slovacko', 'Liberec', 'Sigma Olomouc', 'Hradec Kralove', 'Bohemians', 'Jablonec', 'Ostrava', 'Sparta Prague', 'FK Pardubice', 'Mlada Boleslav', 'Teplice', 'Slavia Prague'}
Number of teams: 16
All teams appear same amount of times in home and away: False
=== Game Balance Check ===

Head-to-head matrix:
                  Bohemians  Ceske Budejovice  Dukla Prague  FK Pardubice  Hradec Kralove  Jablonec  Karvina  Liberec  Mlada Boleslav  Ostrava  Plzen  Sigma Olomouc  Slavia Prague  Slovacko  Sparta Pra

In [59]:
df_only_rounds = filter_playoffs(df_only_rounds, 31)

In [48]:
df_only_rounds

,round,date,team_home,team_away,full_text
0,Round 26,09.03.2025,Omonia Aradippou,AEK Larnaca,09.03.2025 | Omonia Aradippou | AEK Larnaca | ...
1,Round 26,09.03.2025,Pafos,Achnas,09.03.2025 | Pafos | Achnas | 0 | 0
2,Round 26,09.03.2025,Paralimni,Aris,09.03.2025 | Paralimni | Aris | 1 | 2
3,Round 26,08.03.2025,APOEL,Omonia,08.03.2025 | APOEL | Omonia | 2 | 1
4,Round 26,08.03.2025,Apollon,Nea Salamis,08.03.2025 | Apollon | Nea Salamis | 1 | 1
...,...,...,...,...,...
177,Round 1,25.08.2024,Apollon,AEK Larnaca,25.08.2024 | Apollon | AEK Larnaca | 1 | 2
178,Round 1,25.08.2024,Omonia,Achnas,25.08.2024 | Omonia | Achnas | 3 | 0
179,Round 1,24.08.2024,Karmiotissa,Aris,24.08.2024 | Karmiotissa | Aris | 1 | 3
180,Round 1,24.08.2024,Omonia 29th May,AEL Limassol,24.08.2024 | Omonia 29th May | AEL Limassol | ...


In [62]:
df_only_rounds[df_only_rounds["round"] == "Round 1"]

,round,date,team_home,team_away,full_text


In [83]:
df_filtered = filter_on_date(df_only_rounds, "2025-04-20")

C:\Users\simcosyn\AppData\Local\Temp\ipykernel_15052\1965196199.py:8: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')


In [84]:
df_filtered

,round,date,team_home,team_away,full_text
0,Round 30,2025-04-19,FK Pardubice,Sparta Prague,19.04.2025 | FK Pardubice | Sparta Prague | 1 | 2
1,Round 30,2025-04-19,Liberec,Karvina,19.04.2025 | Liberec | Karvina | 2 | 3
2,Round 30,2025-04-19,Mlada Boleslav,Jablonec,19.04.2025 | Mlada Boleslav | Jablonec | 0 | 1
3,Round 30,2025-04-19,Ostrava,Bohemians,19.04.2025 | Ostrava | Bohemians | 1 | 0
4,Round 30,2025-04-19,Plzen,Dukla Prague,19.04.2025 | Plzen | Dukla Prague | 4 | 2
...,...,...,...,...,...
235,Round 1,2024-07-20,Dukla Prague,Plzen,20.07.2024 | Dukla Prague | Plzen | 1 | 3
236,Round 1,2024-07-20,Bohemians,Ostrava,20.07.2024 | Bohemians | Ostrava | 2 | 1
237,Round 1,2024-07-20,Ceske Budejovice,Sigma Olomouc,20.07.2024 | Ceske Budejovice | Sigma Olomouc ...
238,Round 1,2024-07-20,Jablonec,Mlada Boleslav,20.07.2024 | Jablonec | Mlada Boleslav | 2 | 2...


In [85]:
check_rounds(df_filtered)
check_teams(df_filtered)   
check_games(df_filtered)

Rounds: ['Round 30', 'Round 29', 'Round 23', 'Round 28', 'Round 27', 'Round 26', 'Round 21', 'Round 25', 'Round 24', 'Round 22', 'Round 20', 'Round 19', 'Round 18', 'Round 8', 'Round 17', 'Round 15', 'Round 16', 'Round 14', 'Round 13', 'Round 12', 'Round 11', 'Round 10', 'Round 9', 'Round 6', 'Round 7', 'Round 3', 'Round 5', 'Round 4', 'Round 2', 'Round 1']
Only normal rounds: True
All rounds appear same amount of times: True
Teams: {'Plzen', 'Karvina', 'Dukla Prague', 'Ceske Budejovice', 'Liberec', 'Slovacko', 'Hradec Kralove', 'Sigma Olomouc', 'Bohemians', 'Jablonec', 'Ostrava', 'Sparta Prague', 'FK Pardubice', 'Mlada Boleslav', 'Teplice', 'Slavia Prague'}
Number of teams: 16
All teams appear same amount of times in home and away: True
Each team appears 15 times in home and away
=== Game Balance Check ===

Head-to-head matrix:
                  Bohemians  Ceske Budejovice  Dukla Prague  FK Pardubice  Hradec Kralove  Jablonec  Karvina  Liberec  Mlada Boleslav  Ostrava  Plzen  Sigma Ol

In [77]:
df_filtered["team_home"].value_counts()

team_home
FK Pardubice        11
Ostrava             11
Bohemians           11
Sparta Prague       11
Jablonec            11
Mlada Boleslav      10
Plzen               10
Sigma Olomouc       10
Hradec Kralove      10
Slovacko            10
Dukla Prague        10
Liberec              9
Slavia Prague        9
Teplice              9
Ceske Budejovice     9
Karvina              9
Name: count, dtype: int64

In [50]:
# Awrd means forfait.
df_only_rounds[df_only_rounds["team_home"] == "Awrd"]


,round,date,team_home,team_away,full_text
165,Round 3,15.09.2024,Awrd,Apollon,15.09.2024 | Awrd | Apollon | AEL Limassol | 3...


In [52]:
df_only_rounds.loc[165, "team_home"] = "Apollon"
df_only_rounds.loc[165, "team_away"] = "AEL Limassol"

In [87]:
df_filtered.head()

,round,date,team_home,team_away,full_text
0,30,2025-04-19,FK Pardubice,Sparta Prague,19.04.2025 | FK Pardubice | Sparta Prague | 1 | 2
1,30,2025-04-19,Liberec,Karvina,19.04.2025 | Liberec | Karvina | 2 | 3
2,30,2025-04-19,Mlada Boleslav,Jablonec,19.04.2025 | Mlada Boleslav | Jablonec | 0 | 1
3,30,2025-04-19,Ostrava,Bohemians,19.04.2025 | Ostrava | Bohemians | 1 | 0
4,30,2025-04-19,Plzen,Dukla Prague,19.04.2025 | Plzen | Dukla Prague | 4 | 2


In [86]:
df_filtered = remove_round_prefix(df_filtered)

In [88]:
# write away cleaned df and adapt logboek
df_filtered.to_csv(f"./cleaned_data/{filename}", index=False)